# SAM-Med3D Segmentation Visualization

This notebook demonstrates how to:
1. Load random images from each dataset
2. Run SAM-Med3D segmentation
3. Visualize original images and segmentation masks side-by-side with different colors

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import SimpleITK as sitk
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import random
import yaml

# Add project root to path
project_root = Path().cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import SAM-Med3D utilities
from med3pipe.sam.core import (
    build_sam3d_model,
    load_volume_tensor,
    make_pre_transform,
    find_default_sam3d_root
)

## Configuration

In [ ]:
# Load dataset configuration
config_path = project_root / "configs" / "datasets.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# SAM-Med3D configuration
sam3d_root = find_default_sam3d_root()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_size = 128
n_samples_per_dataset = 2  # Number of random images to select from each dataset

print(f"Using device: {device}")
print(f"SAM-Med3D root: {sam3d_root}")
print(f"Available datasets: {list(config['datasets'].keys())}")

## Build SAM-Med3D Model

In [ ]:
# Build the model
model = build_sam3d_model(
    sam3d_root=sam3d_root,
    model_type="vit_b_ori",
    checkpoint=None,  # Add checkpoint path if available
    device=device,
    eval_mode=True
)

print("SAM-Med3D model loaded successfully!")

## Helper Functions

In [ ]:
def find_dataset_images(dataset_config, sam3d_root, project_root):
    """
    Find available images for a dataset.
    Looks in multiple locations: train/val folders, data folder, etc.
    """
    images = []
    labels = []
    
    category = dataset_config['category']
    ct_name = dataset_config['ct_name']
    
    # Check SAM-Med3D data folders
    possible_paths = [
        sam3d_root / 'data' / 'train' / category / ct_name / 'imagesTr',
        sam3d_root / 'data' / 'validation' / category / ct_name / 'imagesVal',
    ]
    
    for path in possible_paths:
        if path.exists():
            imgs = list(path.glob('*.nii.gz'))
            images.extend(imgs)
            
            # Find corresponding labels
            label_path = path.parent / path.name.replace('images', 'labels')
            if label_path.exists():
                for img in imgs:
                    label_file = label_path / img.name
                    if label_file.exists():
                        labels.append(label_file)
    
    return images, labels


def generate_sam_segmentation(model, image_tensor, device):
    """
    Generate segmentation using SAM-Med3D image encoder.
    For full segmentation, we'd need the mask decoder too,
    but here we'll use a simple approach for demonstration.
    """
    with torch.no_grad():
        # Get embeddings from image encoder
        embeddings = model.image_encoder(image_tensor.to(device))
        
        # For demonstration, create a simple segmentation from embeddings
        # In practice, you'd use the full SAM pipeline with prompts
        # Here we'll average across channels and threshold
        B, C, D, H, W = embeddings.shape
        
        # Upsample to original size
        if D < img_size or H < img_size or W < img_size:
            embeddings_up = torch.nn.functional.interpolate(
                embeddings.mean(dim=1, keepdim=True),
                size=(img_size, img_size, img_size),
                mode='trilinear',
                align_corners=False
            )
        else:
            embeddings_up = embeddings.mean(dim=1, keepdim=True)
        
        # Simple thresholding for demonstration
        seg = (embeddings_up > embeddings_up.mean()).float()
        
    return seg.cpu().squeeze().numpy()


def plot_slices(image_vol, seg_vol, gt_mask_vol=None, title="", n_slices=5):
    """
    Plot central slices from 3D volumes.
    """
    D, H, W = image_vol.shape
    
    # Select evenly spaced slice indices
    slice_indices = np.linspace(D//4, 3*D//4, n_slices, dtype=int)
    
    n_cols = 3 if gt_mask_vol is not None else 2
    fig, axes = plt.subplots(n_slices, n_cols, figsize=(4*n_cols, 3*n_slices))
    
    if n_slices == 1:
        axes = axes.reshape(1, -1)
    
    for i, slice_idx in enumerate(slice_indices):
        # Original image
        axes[i, 0].imshow(image_vol[slice_idx], cmap='gray')
        axes[i, 0].set_title(f"Original (slice {slice_idx})")
        axes[i, 0].axis('off')
        
        # Overlay SAM segmentation
        axes[i, 1].imshow(image_vol[slice_idx], cmap='gray')
        # Overlay segmentation in red with transparency
        seg_overlay = np.ma.masked_where(seg_vol[slice_idx] == 0, seg_vol[slice_idx])
        axes[i, 1].imshow(seg_overlay, cmap='Reds', alpha=0.5, vmin=0, vmax=1)
        axes[i, 1].set_title(f"SAM Segmentation (slice {slice_idx})")
        axes[i, 1].axis('off')
        
        # Ground truth if available
        if gt_mask_vol is not None:
            axes[i, 2].imshow(image_vol[slice_idx], cmap='gray')
            # Overlay GT in green with transparency
            gt_overlay = np.ma.masked_where(gt_mask_vol[slice_idx] == 0, gt_mask_vol[slice_idx])
            axes[i, 2].imshow(gt_overlay, cmap='Greens', alpha=0.5, vmin=0, vmax=1)
            axes[i, 2].set_title(f"Ground Truth (slice {slice_idx})")
            axes[i, 2].axis('off')
    
    fig.suptitle(title, fontsize=16, y=0.995)
    plt.tight_layout()
    return fig

## Process Datasets and Generate Segmentations

In [ ]:
# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

pre_transform = make_pre_transform(img_size=img_size)

# Process each dataset
for dataset_name, dataset_config in config['datasets'].items():
    print(f"\n{'='*60}")
    print(f"Processing dataset: {dataset_name.upper()}")
    print(f"{'='*60}")
    
    # Find available images
    images, labels = find_dataset_images(dataset_config, sam3d_root, project_root)
    
    if not images:
        print(f"⚠️  No images found for {dataset_name}. Skipping...")
        continue
    
    print(f"Found {len(images)} images")
    
    # Select random samples
    n_samples = min(n_samples_per_dataset, len(images))
    selected_indices = random.sample(range(len(images)), n_samples)
    
    for idx in selected_indices:
        img_path = images[idx]
        print(f"\nProcessing: {img_path.name}")
        
        try:
            # Load image
            image_tensor = load_volume_tensor(img_path, pre_transform=pre_transform)
            
            # Generate segmentation
            print("  Generating SAM segmentation...")
            sam_seg = generate_sam_segmentation(model, image_tensor, device)
            
            # Load ground truth if available
            gt_mask = None
            if idx < len(labels) and labels[idx].exists():
                print("  Loading ground truth mask...")
                gt_sitk = sitk.ReadImage(str(labels[idx]))
                gt_mask = sitk.GetArrayFromImage(gt_sitk)
                # Apply same preprocessing
                import torchio as tio
                gt_tensor, _ = tio.data.io.sitk_to_nib(gt_sitk)
                subj = tio.Subject(label=tio.LabelMap(tensor=gt_tensor))
                subj = pre_transform(subj)
                gt_mask = subj.label.data.squeeze().numpy()
                gt_mask = (gt_mask > 0).astype(float)
            
            # Get image array for visualization
            image_np = image_tensor.squeeze().cpu().numpy()
            
            # Plot
            print("  Plotting...")
            fig = plot_slices(
                image_np,
                sam_seg,
                gt_mask,
                title=f"{dataset_name.upper()} - {img_path.stem}",
                n_slices=5
            )
            plt.show()
            
        except Exception as e:
            print(f"  ❌ Error processing {img_path.name}: {e}")
            continue

print("\n" + "="*60)
print("Processing complete!")
print("="*60)

## Example with Test Data

If no dataset images are available, let's use the toy test data:

In [ ]:
# Use toy test data if available
test_data_path = sam3d_root / 'test_data' / 'amos_val_toy_data'

if test_data_path.exists():
    print(f"Using test data from: {test_data_path}")
    
    images_path = test_data_path / 'imagesVa'
    labels_path = test_data_path / 'labelsVa'
    
    test_images = list(images_path.glob('*.nii.gz'))
    
    if test_images:
        print(f"Found {len(test_images)} test images")
        
        for img_path in test_images[:2]:  # Process first 2 images
            print(f"\nProcessing test image: {img_path.name}")
            
            try:
                # Load image
                image_tensor = load_volume_tensor(img_path, pre_transform=pre_transform)
                
                # Generate segmentation
                print("  Generating SAM segmentation...")
                sam_seg = generate_sam_segmentation(model, image_tensor, device)
                
                # Load ground truth if available
                gt_mask = None
                label_path = labels_path / img_path.name
                if label_path.exists():
                    print("  Loading ground truth mask...")
                    import torchio as tio
                    gt_sitk = sitk.ReadImage(str(label_path))
                    gt_tensor, _ = tio.data.io.sitk_to_nib(gt_sitk)
                    subj = tio.Subject(label=tio.LabelMap(tensor=gt_tensor))
                    subj = pre_transform(subj)
                    gt_mask = subj.label.data.squeeze().numpy()
                    gt_mask = (gt_mask > 0).astype(float)
                
                # Get image array
                image_np = image_tensor.squeeze().cpu().numpy()
                
                # Plot
                print("  Plotting...")
                fig = plot_slices(
                    image_np,
                    sam_seg,
                    gt_mask,
                    title=f"Test Data - {img_path.stem}",
                    n_slices=5
                )
                plt.show()
                
            except Exception as e:
                print(f"  ❌ Error: {e}")
                import traceback
                traceback.print_exc()
else:
    print("No test data found.")

## Notes

- **Red overlay**: SAM-Med3D generated segmentation
- **Green overlay**: Ground truth segmentation (when available)
- This notebook uses a simplified segmentation approach for demonstration
- For production use, integrate the full SAM pipeline with prompt engineering
- Adjust `n_samples_per_dataset` and `n_slices` for different visualization needs
- When dataset images are available in `data/gist/`, `data/lipo/`, etc., they will be automatically discovered